In [ ]:
import concurrent.futures
import pandas as pd
import numpy as np
import dateparser
import os
import re

#Código para analizar los archivos de aclaraciones de calificaciones 

ruta_general = "C:/Users/magal/Desktop/ACLARACIONES"
rutas = []
rutas_csv = []

def load_csv(file_path):
    """Carga un archivo csv en un DataFrame"""
    try:
        df = pd.read_csv(file_path)

        patron_fecha = r'ACLARACIONES\\(\d{2}\.\d{2}\.\d{2})\\'
        fecha_match = re.search(patron_fecha, file_path)
        fecha = fecha_match.group(1) if fecha_match else None
        df["FechaRuta"] = pd.to_datetime(fecha, format="%d.%m.%y", errors="coerce")
        #df["FechaRuta"] = fecha

        tipo_match = re.search(r'\\(ORD|RECU)\\', file_path)
        tipo = tipo_match.group(1) if tipo_match else None
        df["Asignacion"] = tipo



        #patron = r'\\([A-Za-z]+)\\'
        #coincidencias = re.findall(patron, file_path)

        #mes = coincidencias[-1]
        #df["Mes"] = mes

        return df
    except Exception as e:
        print(f"Error al cargar {file_path}: {e}")
        return None  # En caso de error, retornamos None

if __name__ == "__main__":
    print("Inicia programa")
    for fecha in os.listdir(ruta_general): 
        ruta_fecha = os.path.join(ruta_general, fecha)  
        if os.path.isdir(ruta_fecha):  
            for asignacion in os.listdir(ruta_fecha):
                asignacion_path = os.path.join(ruta_fecha, asignacion) 
                #print(asignacion_path)  
                if os.path.isdir(asignacion_path):  
                    #rutas.append(asignacion_path)
                    for archivo in os.listdir(asignacion_path):
                        #print(archivo)
                        if archivo.endswith(".csv"):
                            rutas_csv.append(os.path.join(asignacion_path, archivo))


    #for rutas in rutas_csv:
        #print(rutas)

    with concurrent.futures.ThreadPoolExecutor() as executor:
        dataframes = list(executor.map(load_csv, rutas_csv))
    
    df_final = pd.concat(dataframes, ignore_index=True)

    #print(df_final.head())
    df_final = df_final.rename(columns={'FechaRuta': 'Fecha cierre'})

    df_final['Módulo'] = df_final['Grupo'].str.extract(r'(\d+)')

    df_final['Actividad'] = (
        df_final['Actividad']
            .str.replace('AI5', 'Actividad integradora 5', regex=False)
            .str.replace('AI4', 'Actividad integradora 4', regex=False)
            .str.replace('AI6', 'Actividad integradora 6', regex=False)
            .str.replace('AI3', 'Actividad integradora 3', regex=False)
            .str.replace('AI2', 'Actividad integradora 2', regex=False)
            .str.replace('A1', 'Actividad integradora 1', regex=False)
            .str.replace('AI1', 'Actividad integradora 1', regex=False)
            .str.replace('PI', 'Proyecto integrador', regex=False)
            .str.replace('Actividad Integradora', 'Actividad integradora', regex=False)
            .str.replace(':', '.', regex=False)
    )

    #Extraer número de actividad
    df_final['Numero act'] = df_final['Actividad'].str.extract(r'(\d+)')[0] \
        .fillna(df_final['Actividad'].str.extract(r'([A-Za-zÁÉÍÓÚáéíóúñÑ]+)')[0])
    df_final['Tipo act'] = df_final['Actividad'].str.extract(
        r'^([A-Za-zÁÉÍÓÚáéíóúñÑ ]+)\.?'  # letras y espacios hasta un posible punto
    )

    df_final['Tipo act'] = df_final['Tipo act'].str.rstrip()

    df_final['Numero act'] = (
        df_final['Numero act']
            .str.replace('Proyecto', '7', regex=False)
            .str.replace('Fase', '7', regex=False)
    )        

    df_final.loc[1074, 'Numero act'] = '7'

    # Limpiando Razón ADC
    df_final['Razón ADC'] = df_final['Razón ADC'].str.strip()

    df_final['Categoría ADC'] = (
        df_final['Razón ADC']
        .str.strip()
        .str.split('-', n=1)
        .str[0]
        .str.strip()
        .replace('', '-')  
        .fillna('-')            
    )
    df_final['Razón'] = (
        df_final['Razón ADC']
        .str.split('-', n=1)
        .str[1]               
        .str.strip()         
        .replace(['', '-'], 'Sin detalles')  
        .fillna('Sin detalles')             
    )

    col = df_final.pop('Razón')  
    df_final['Razón'] = col

    frecuencia_modulo = df_final['Módulo'].value_counts()
    print(frecuencia_modulo)

    frecuencia_tipoact = df_final['Tipo act'].value_counts()
    print(frecuencia_tipoact)

    frecuencia_satis = df_final['Satisfacción de la aclaración'].value_counts()
    print(frecuencia_satis)

    frecuencia_razonadc = df_final['Categoría ADC'].value_counts()
    print(frecuencia_razonadc)

    frecuencia_razon = df_final['Razón'].value_counts()
    print(frecuencia_razon)

    #Obteniendo la frecuencia por actividad y módulo

    frecuencia_act_por_modulo = pd.crosstab(
        df_final['Numero act'],    
        df_final['Módulo']
    )

    # Dar nombre a índices y columnas
    frecuencia_act_por_modulo.index.name = 'Número de actividad'
    frecuencia_act_por_modulo.columns.name = 'Módulo'

    # Mostrar toda la tabla sin recortes
    with pd.option_context('display.max_rows', None, 'display.max_columns', None):
        print(frecuencia_act_por_modulo)

"""    #Convertir fechas para calculos
    columnas_fechas = ['Fecha aclaración emitida', 'Fecha revisión emitida', 'Fecha retroalimentación asesor', 'Fecha resolución supervisor']
    for col in columnas_fechas:
        df_final[col] = df_final[col].apply(lambda x: dateparser.parse(x, languages=['es']))"""


In [ ]:
#df_final.to_excel(r"C:\Users\magal\Desktop\ACLARACIONES\Aclaraciones.xlsx", index=False)


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("C:/Users/magal/Desktop/ACLARACIONES/16.10.25/ORD/Módulo 15 G62__reporte.csv")
"""df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
estado = df.loc[:, 'Estado']
print(estado)
df.info()
missing_count = df.isnull().sum()"""
df_aclaraciones = df[['Grupo', 'Actividad', 'Razón', 'Estado', 'Fase', 'Satisfacción de la aclaración', 'Razón ADC']]
df_aclaraciones['Módulo'] = df_aclaraciones['Grupo'].str.extract(r'(\d+)')
df_aclaraciones['AI'] = df_aclaraciones['Actividad'].str.extract(r'(\d+)')
df_aclaraciones = df_aclaraciones[['Módulo', 'Grupo', 'Actividad', 'AI', 'Estado', 'Fase', 'Satisfacción de la aclaración', 'Razón', 'Razón ADC']]

frecuencia_modulo = df_aclaraciones['Módulo'].value_counts()
df_frecuencias = pd.DataFrame({
    'Módulo': frecuencia_modulo.index,
    'Frecuencia': frecuencia_modulo.values,
    'Texto': [f'Frecuencia aclaraciones del módulo "{mod}" es {freq}' for mod, freq in frecuencia_modulo.items()]
})
#print(df_frecuencias)

frecuencia_actividad = df_aclaraciones['AI'].value_counts()
df_frecuenciaAI = pd.DataFrame({
    'Actividad': frecuencia_actividad.index,
    'Frecuencia': frecuencia_actividad.values,
    'Texto': [f'Frecuencia aclaraciones en actividad "{mod}" es {freq}' for mod, freq in frecuencia_actividad.items()]
})
#print(df_frecuenciaAI)

frecuencia_estado = df_aclaraciones['Estado'].value_counts()
df_frecuencia_edo = pd.DataFrame({
    'Estado': frecuencia_estado.index,
    'Frecuencia': frecuencia_estado.values,
    'Texto': [f'Frecuencia aclaraciones estado "{mod}" es {freq}' for mod, freq in frecuencia_estado.items()]
})
print(df_frecuencia_edo)

frecuencia_satisfaccion = df_aclaraciones['Satisfacción de la aclaración'].value_counts()
df_frecuencia_satisfaccion = pd.DataFrame({
    'Satisfacción': frecuencia_satisfaccion.index,
    'Frecuencia': frecuencia_satisfaccion.values,
    'Texto': [f'Frecuencia aclaraciones "{mod}" es {freq}' for mod, freq in frecuencia_satisfaccion.items()]
})
print(df_frecuencia_satisfaccion)
